# Deep Hedging in Incomplete Markets
### ML in Finance — Group Project
*Based on: Fecamp, Mikael & Warin (2019)*

---
**Run cells top-to-bottom.** Each section is self-contained.
Install dependencies once with the cell below, then restart the kernel.

## 0 · Install dependencies

In [ ]:
# Run once, then restart kernel if needed
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "torch", "numpy", "matplotlib", "scipy"])
print("Done ✓")

## 1 · Config
All hyperparameters in one place. **Edit this cell to change any experiment setting.**

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
torch.manual_seed(42)
np.random.seed(42)

@dataclass
class Config:
    # --- Market ---
    S0: float = 1.0
    K: float  = 1.0
    T: float  = 1/12       # 1-month maturity
    sigma: float = 0.2     # annualised volatility
    mu: float    = 0.0     # drift (0 = risk-neutral martingale)
    r: float     = 0.0     # risk-free rate

    # --- Hedging ---
    N: int   = 30          # number of hedging dates
    liquidity: float = 1.0 # max position per step (high = unconstrained)

    # --- Training ---
    n_train: int   = 50_000
    n_test: int    = 100_000
    batch_size: int = 256
    n_iter: int    = 20_000
    lr: float      = 1e-3

    # --- Network ---
    lstm_hidden: int = 64
    ff_layers: int   = 2
    ff_width: int    = 32

    # --- Loss: "mse" | "asymmetric" | "m2m4" | "cvar" ---
    loss_name: str   = "mse"
    alpha: float     = 1.5   # penalty for asymmetric / m2m4
    cvar_level: float = 0.95 # CVaR confidence level

    # --- Transaction costs ---
    use_transaction_costs: bool = False
    cost_per_unit: float = 0.02
    tc_alpha: float      = 0.5   # 0=cost only, 1=variance only

    # --- Reproducibility ---
    seed: int = 42

    dt: float = field(init=False)
    def __post_init__(self): self.dt = self.T / self.N
    def summary(self):
        print(f"Config | T={self.T:.3f}  N={self.N}  sigma={self.sigma}  "
              f"loss={self.loss_name}  tc={self.use_transaction_costs}")

cfg = Config()
cfg.summary()
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2 · Market simulation (GBM)
Geometric Brownian Motion — equity replacement for the paper's electricity model.
`paths` has shape **(n_paths, N+1)**: column 0 = S₀, column -1 = S_T.

In [ ]:
class GBMMarket:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def simulate(self, n_paths: int, device: str = "cpu") -> torch.Tensor:
        cfg = self.cfg
        torch.manual_seed(cfg.seed)
        Z = torch.randn(n_paths, cfg.N, device=device)
        log_return = (cfg.mu - 0.5*cfg.sigma**2)*cfg.dt + cfg.sigma*np.sqrt(cfg.dt)*Z
        log_prices = torch.cat([torch.zeros(n_paths, 1, device=device),
                                 torch.cumsum(log_return, dim=1)], dim=1)
        return cfg.S0 * torch.exp(log_prices)

# Quick visual check
market = GBMMarket(cfg)
sample = market.simulate(200).numpy()

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(sample[:50].T, alpha=0.4, linewidth=0.8)
ax.set(xlabel="Time step", ylabel="Price", title=f"GBM paths  (T={cfg.T:.2f}, σ={cfg.sigma}, N={cfg.N})")
plt.tight_layout(); plt.show()
print(f"Path tensor shape: {sample.shape}")

## 3 · Payoffs & Black-Scholes benchmark
The BS delta is your **ground truth** — in the unconstrained, no-cost case the LSTM
should get close to it. If it doesn't, something is wrong with the training.

In [ ]:
def payoff_call(S_T: torch.Tensor, K: float) -> torch.Tensor:
    return torch.clamp(S_T - K, min=0.0)

def payoff_put(S_T: torch.Tensor, K: float) -> torch.Tensor:
    return torch.clamp(K - S_T, min=0.0)

def payoff_spread(S1_T, S2_T, K):
    return torch.clamp(S1_T - S2_T - K, min=0.0)

def bs_delta(S, K, T_rem, sigma, r=0.0):
    """Analytical Black-Scholes delta for a call."""
    from torch.distributions import Normal
    eps = 1e-8
    d1 = (torch.log(S/K) + (r + 0.5*sigma**2)*T_rem) / (sigma*torch.sqrt(T_rem) + eps)
    return Normal(0, 1).cdf(d1)

# Sanity check: BS delta at-the-money with 1 month left
S = torch.tensor([[1.0]])
T_rem = torch.tensor([[1/12]])
delta = bs_delta(S, cfg.K, T_rem, cfg.sigma)
print(f"BS delta (ATM, 1 month): {delta.item():.4f}  — should be ~0.52 for ATM call")

## 4 · Loss functions
| Name | Formula | Paper ref |
|------|---------|-----------|
| MSE | E[Y²] | Eq. (4) |
| Asymmetric | E[(1+α)Y²·1_{Y≤0} + Y²·1_{Y>0}] | Eq. (5) |
| Moment2/4 | E[Y²·1_{Y≥0}] + α·E[Y⁴·1_{Y≤0}] | Eq. (6) |
| **CVaR** | E[−Y \| −Y > VaR_α] | **Your addition** |

`Y = X_T − g(S_T)` is the hedging P&L. Losses = negative Y.

In [ ]:
class MSELoss(nn.Module):
    def forward(self, pnl): return pnl.pow(2).mean()

class AsymmetricLoss(nn.Module):
    def __init__(self, alpha=1.5):
        super().__init__(); self.alpha = alpha
    def forward(self, pnl):
        loss_side = (1 + self.alpha) * pnl.pow(2) * (pnl <= 0).float()
        gain_side = pnl.pow(2) * (pnl > 0).float()
        return (loss_side + gain_side).mean()

class Moment2Moment4Loss(nn.Module):
    def __init__(self, alpha=2.0):
        super().__init__(); self.alpha = alpha
    def forward(self, pnl):
        return (pnl.pow(2)*(pnl>=0).float()).mean() + self.alpha*(pnl.pow(4)*(pnl<=0).float()).mean()

class CVaRLoss(nn.Module):
    """YOUR ADDITION — Expected Shortfall (Basel III / Solvency II)."""
    def __init__(self, level=0.95):
        super().__init__(); self.level = level
    def forward(self, pnl):
        losses = -pnl
        k = max(1, int(np.ceil((1 - self.level) * losses.shape[0])))
        sorted_losses, _ = torch.sort(losses, descending=True)
        return sorted_losses[:k].mean()

class TransactionCostCriterion(nn.Module):
    """Eq. (22): (1-α)·E[TC] + α·√E[(X_T - g)²]"""
    def __init__(self, alpha=0.5):
        super().__init__(); self.alpha = alpha
    def forward(self, pnl, tc):
        return (1 - self.alpha)*tc.mean() + self.alpha*torch.sqrt(pnl.pow(2).mean())

def get_loss(cfg):
    return {"mse": MSELoss(),
            "asymmetric": AsymmetricLoss(cfg.alpha),
            "m2m4": Moment2Moment4Loss(cfg.alpha),
            "cvar": CVaRLoss(cfg.cvar_level)}[cfg.loss_name]

print("Loss functions defined ✓")

## 5 · Neural network architectures
### Augmented LSTM (paper's best — Figure 4)
- **LSTM cell**: processes one time step at a time, weights shared across all steps
- **Feedforward head**: projects LSTM output → unbounded position change
- **tanh clipping**: constrains Δ change to `[-liquidity, +liquidity]` (Eq. 13)

The `input_size=3` variant adds α as a feature for Pareto frontier training.

In [ ]:
class AugmentedLSTMHedger(nn.Module):
    def __init__(self, cfg: Config, input_size: int = 2):
        super().__init__()
        h = cfg.lstm_hidden

        # LSTM cell — weights SHARED across all time steps
        self.lstm_cell = nn.LSTMCell(input_size=input_size, hidden_size=h)

        # Feedforward projection after LSTM output
        layers = []
        in_dim = h
        for _ in range(cfg.ff_layers):
            layers += [nn.Linear(in_dim, cfg.ff_width), nn.ReLU()]
            in_dim = cfg.ff_width
        layers.append(nn.Linear(in_dim, 1))
        self.ff = nn.Sequential(*layers)

        # Trainable premium p (optimised jointly with hedge)
        self.log_premium = nn.Parameter(torch.tensor(0.0))

    def forward(self, paths_norm: torch.Tensor, cfg: Config,
                alpha_val: Optional[float] = None) -> dict:
        batch, device = paths_norm.shape[0], paths_norm.device

        h_t = torch.zeros(batch, cfg.lstm_hidden, device=device)
        c_t = torch.zeros(batch, cfg.lstm_hidden, device=device)
        delta_prev = torch.zeros(batch, device=device)
        pnl = torch.zeros(batch, device=device)
        tc  = torch.zeros(batch, device=device)
        deltas = []

        for j in range(cfg.N):
            # Build input: (S̃_t, t/T) — or (S̃_t, t/T, α) for Pareto
            S_j   = paths_norm[:, j].unsqueeze(1)
            t_feat = torch.full((batch, 1), j/cfg.N, device=device)
            if alpha_val is not None:
                x_j = torch.cat([S_j, t_feat,
                                  torch.full((batch,1), alpha_val, device=device)], dim=1)
            else:
                x_j = torch.cat([S_j, t_feat], dim=1)

            # LSTM step
            h_t, c_t = self.lstm_cell(x_j, (h_t, c_t))

            # Project → position change, then clamp via tanh  [Eq. 13]
            C_hat   = self.ff(h_t).squeeze(1)
            delta_t = delta_prev + cfg.liquidity * torch.tanh(C_hat)
            deltas.append(delta_t.unsqueeze(1))

            # Accumulate P&L:  Δ_t · (F_{t+1} - F_t)
            dF  = paths_norm[:, j+1] - paths_norm[:, j]
            pnl = pnl + delta_t * dF

            # Transaction costs
            if cfg.use_transaction_costs:
                tc = tc + torch.abs(delta_t - delta_prev) * cfg.cost_per_unit

            delta_prev = delta_t

        premium = self.log_premium.expand(batch)
        return {"pnl": pnl + premium,
                "delta": torch.cat(deltas, dim=1),
                "tc": tc,
                "premium": self.log_premium}

# Count parameters
model_check = AugmentedLSTMHedger(cfg)
n_params = sum(p.numel() for p in model_check.parameters() if p.requires_grad)
print(f"Augmented LSTM | trainable parameters: {n_params:,}")

## 6 · DeepHedger — training wrapper
`hedger.train()` runs **Algorithm 1** of the paper:
mini-batch Adam gradient descent on the global loss.

In [ ]:
class DeepHedger:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.market = GBMMarket(cfg)
        self.model  = AugmentedLSTMHedger(cfg).to(self.device)
        self.loss_fn = get_loss(cfg)

        # Pre-simulate all paths once (fast batching during training)
        self.train_paths = self.market.simulate(cfg.n_train, self.device)
        self.test_paths  = self.market.simulate(cfg.n_test,  self.device)

        # Normalise: S̃_t = (S_t - mean) / std   (per time step)
        self._mean = self.train_paths.mean(0, keepdim=True)
        self._std  = self.train_paths.std(0,  keepdim=True).clamp(min=1e-8)

    def _norm(self, p): return (p - self._mean) / self._std

    def _pnl(self, paths):
        out  = self.model(self._norm(paths), self.cfg)
        g_ST = payoff_call(paths[:, -1], self.cfg.K)
        return out["pnl"] - g_ST, out["tc"], out

    def train(self):
        cfg = self.cfg
        opt = torch.optim.Adam(self.model.parameters(), lr=cfg.lr)
        sch = torch.optim.lr_scheduler.StepLR(opt, step_size=cfg.n_iter//3, gamma=0.5)

        self.train_losses, self.test_losses, self.test_iters = [], [], []
        best_loss, best_state = float("inf"), None

        print(f"Training | loss={cfg.loss_name} | {cfg.n_iter} iters | device={self.device}")
        for it in range(cfg.n_iter):
            self.model.train()
            idx = torch.randint(0, cfg.n_train, (cfg.batch_size,))
            pnl, tc, _ = self._pnl(self.train_paths[idx])

            if cfg.use_transaction_costs:
                loss = TransactionCostCriterion(cfg.tc_alpha)(pnl, tc)
            else:
                loss = self.loss_fn(pnl)

            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            opt.step(); sch.step()
            self.train_losses.append(loss.item())

            if (it+1) % 1000 == 0:
                self.model.eval()
                with torch.no_grad():
                    pnl_t, tc_t, _ = self._pnl(self.test_paths)
                    tl = self.loss_fn(pnl_t).item()
                self.test_losses.append(tl)
                self.test_iters.append(it+1)
                print(f"  [{it+1:>6}] train={loss.item():.3e}  test={tl:.3e}")
                if tl < best_loss:
                    best_loss  = tl
                    best_state = {k: v.clone() for k,v in self.model.state_dict().items()}

        if best_state:
            self.model.load_state_dict(best_state)
            print(f"  Best model restored (test loss={best_loss:.3e})")

    def evaluate(self) -> dict:
        self.model.eval()
        with torch.no_grad():
            pnl, tc, out = self._pnl(self.test_paths)
        arr = pnl.cpu().numpy()
        return {"mse": float(np.mean(arr**2)),
                "cvar_95": float(np.mean(arr[arr < np.percentile(arr, 5)])),
                "mean": float(arr.mean()), "std": float(arr.std()),
                "pnl_array": arr,
                "delta_array": out["delta"].cpu().numpy(),
                "premium": float(out["premium"].item()),
                "tc_mean": float(tc.mean().item())}

print("DeepHedger defined ✓")

## 7 · Plotting helpers

In [ ]:
def plot_loss_curves(hedger, title=""):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.semilogy(hedger.train_losses, alpha=0.5, label="Train", linewidth=0.8)
    ax.semilogy(hedger.test_iters, hedger.test_losses, label="Test", linewidth=2)
    ax.set(xlabel="Iteration", ylabel="Loss (log scale)",
           title=f"Loss curves — {hedger.cfg.loss_name} {title}")
    ax.legend(); plt.tight_layout(); plt.show()

def plot_pnl_distributions(results_dict: dict, title=""):
    """Overlay P&L density for multiple loss functions — Figures 9/10/11."""
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    ls_cycle = ["-", "--", ":", "-."]
    for ax, (label, xlim) in zip(axes, [
        ("Full distribution", None),
        ("Left tail (losses)", (-4, -0.5)),
        ("Right tail (gains)", (0.5, 4)),
    ]):
        for (name, res), ls in zip(results_dict.items(), ls_cycle):
            arr = res["pnl_array"]
            ax.hist(arr, bins=300, density=True, histtype="step",
                    label=name, linestyle=ls, linewidth=1.5)
        ax.set(xlabel="P&L  (X_T − g(S_T))", ylabel="Density", title=label)
        if xlim: ax.set_xlim(xlim)
        ax.legend(fontsize=8)
    plt.suptitle(f"P&L Distributions {title}", y=1.02)
    plt.tight_layout(); plt.show()

def plot_deltas(hedger: DeepHedger, n_paths: int = 5):
    """Hedge delta over time for sample paths — Figures 6/7/8."""
    arr = hedger.evaluate()["delta_array"][:n_paths]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for i, d in enumerate(arr):
        ax.plot(d, marker="o", markersize=3, label=f"Path {i+1}")
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set(xlabel="Time step j", ylabel="Δ_j (hedge position)",
           title=f"Hedge deltas — {hedger.cfg.loss_name}")
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()

def plot_pareto(curve: dict):
    """Figure 12: MSE vs mean transaction cost for varying α."""
    fig, ax = plt.subplots(figsize=(6, 5))
    sc = ax.scatter(curve["mse"], curve["tc_mean"],
                    c=curve["alpha"], cmap="viridis", zorder=5)
    for i, a in enumerate(curve["alpha"]):
        if abs(a - round(a, 1)) < 0.03:
            ax.annotate(f"α={a:.1f}",
                (curve["mse"][i], curve["tc_mean"][i]),
                fontsize=8, xytext=(5, 4), textcoords="offset points")
    plt.colorbar(sc, label="α")
    ax.set(xlabel="MSE (hedging variance)", ylabel="Mean transaction cost",
           title="Pareto frontier: risk vs transaction cost")
    plt.tight_layout(); plt.show()

def print_table(results: dict):
    print(f"\n{'Method':<22} {'MSE':>12} {'CVaR-95%':>12} {'Premium':>10}")
    print("-" * 58)
    for name, r in results.items():
        print(f"{name:<22} {r['mse']:>12.4e} {r['cvar_95']:>12.4e} {r.get('premium', 0):>10.4f}")

print("Plotting helpers defined ✓")

## 8 · Quick sanity check ✅
Run this first — it trains for only 500 iterations to verify everything works.
**Expected**: loss should visibly decrease. MSE will still be high (not converged yet).

In [ ]:
cfg_quick = Config(loss_name="mse", n_iter=500, n_train=5_000, n_test=10_000,
                   batch_size=128, N=14)
hedger_quick = DeepHedger(cfg_quick)
hedger_quick.train()

res_quick = hedger_quick.evaluate()
print(f"\nSanity check MSE : {res_quick['mse']:.4e}")
print(f"CVaR-95%         : {res_quick['cvar_95']:.4e}")
print(f"Learned premium  : {res_quick['premium']:.4f}")

plot_loss_curves(hedger_quick, title="(quick run)")
plot_deltas(hedger_quick)

## 9 · Full experiment — compare loss functions
**⚠️ This takes ~5–20 min on CPU depending on `n_iter`.**
Runs MSE, Asymmetric, and CVaR and overlays their P&L distributions.
This is the core result for **Member 2** and the report.

In [ ]:
LOSS_NAMES = ["mse", "asymmetric", "cvar"]

all_results = {}
for loss_name in LOSS_NAMES:
    print(f"\n{'='*55}\n  Loss = {loss_name.upper()}\n{'='*55}")
    cfg_exp = Config(loss_name=loss_name, n_iter=20_000,
                     n_train=50_000, n_test=100_000)
    h = DeepHedger(cfg_exp)
    h.train()
    res = h.evaluate()
    all_results[loss_name.upper()] = res
    plot_loss_curves(h)

plot_pnl_distributions(all_results, title="— full training")
print_table(all_results)

## 10 · Maturity sweep
**Your original experiment** — study how accuracy degrades as maturity T (and number
of hedging dates N) increases. Reports MSE for each configuration.

In [ ]:
maturities = [1/12, 3/12, 6/12]   # 1 month, 3 months, 6 months
sweep = {}

for T in maturities:
    N   = max(14, int(T * 365 / 7))   # approx weekly hedging dates
    cfg_sweep = Config(T=T, N=N, loss_name="mse", n_iter=10_000,
                       n_train=30_000, n_test=50_000)
    print(f"\nT={T:.2f} yr | N={N} steps")
    h = DeepHedger(cfg_sweep)
    h.train()
    r = h.evaluate()
    sweep[f"T={T:.2f} N={N}"] = r

print("\n-- Maturity sweep results --")
for k, r in sweep.items():
    print(f"  {k:<20}  MSE={r['mse']:.4e}  CVaR-95%={r['cvar_95']:.4e}")

## 11 · Transaction costs & Pareto frontier
Replicates **Section 7** of the paper. One network trained for all α ∈ [0,1] simultaneously.
The curve shows the trade-off between hedging variance and transaction cost.

In [ ]:
class ParetoHedger(DeepHedger):
    """Extends DeepHedger: adds α as a network input for joint Pareto training."""
    def __init__(self, cfg):
        cfg.use_transaction_costs = True
        super().__init__(cfg)
        # Override model with input_size=3 (S̃, t/T, α)
        self.model = AugmentedLSTMHedger(cfg, input_size=3).to(self.device)

    def train(self, n_iter=50_000):
        cfg = self.cfg
        opt = torch.optim.Adam(self.model.parameters(), lr=cfg.lr)
        print(f"Pareto training | {n_iter} iters | randomly sampling α each step")
        for it in range(n_iter):
            self.model.train()
            alpha_val = float(torch.rand(1).item())
            idx = torch.randint(0, cfg.n_train, (cfg.batch_size,))
            out = self.model(self._norm(self.train_paths[idx]), cfg, alpha_val=alpha_val)
            pnl = out["pnl"] - payoff_call(self.train_paths[idx, -1], cfg.K)
            loss = TransactionCostCriterion(alpha_val)(pnl, out["tc"])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            opt.step()
            if (it+1) % 10_000 == 0:
                print(f"  [{it+1}] α={alpha_val:.2f}  loss={loss.item():.3e}")

    def pareto_point(self, alpha_val: float) -> dict:
        self.model.eval()
        with torch.no_grad():
            out = self.model(self._norm(self.test_paths), self.cfg, alpha_val=alpha_val)
            pnl = (out["pnl"] - payoff_call(self.test_paths[:, -1], self.cfg.K)).cpu().numpy()
        return {"alpha": alpha_val, "mse": float(np.mean(pnl**2)),
                "tc_mean": float(out["tc"].cpu().numpy().mean())}

    def compute_curve(self, n_pts=20):
        pts = [self.pareto_point(a) for a in np.linspace(0, 1, n_pts)]
        return {"alpha":   [p["alpha"]   for p in pts],
                "mse":     [p["mse"]     for p in pts],
                "tc_mean": [p["tc_mean"] for p in pts]}

# --- Run Pareto training ---
cfg_pareto = Config(n_iter=50_000, use_transaction_costs=True,
                    n_train=50_000, n_test=100_000)
pareto = ParetoHedger(cfg_pareto)
pareto.train(n_iter=50_000)

curve = pareto.compute_curve(n_pts=20)
plot_pareto(curve)

## 12 · Final results summary
Collect all results from the experiments above and print the report-ready table.

In [ ]:
# Collect everything into one table
# (assumes you ran sections 9 and 10 above)
summary = {**all_results}

# Add maturity sweep MSE summary
for k, r in sweep.items():
    summary[f"LSTM MSE [{k}]"] = r

print_table(summary)
print("\n✓ All results printed — copy into your report Table 3 equivalent.")